In [1]:
# IMPORT LIBRARY
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
# MEMUAT HASIL PENILAIAN HARGA OPSI PUT MODEL BLACk-SCHOLES
df_bs = pd.read_csv("Penilaian_Model_Black-Scholes.csv")

columns_to_keep = [
    "ID_OPTION",
    "QUOTE_DATE",
    "BS_PRICE"
]
df_bs = df_bs[columns_to_keep]

In [3]:
# MEMUAT HASIL PENILAIAN HARGA OPSI PUT MODEL LSTM
dfs = []

for i in range(1, 13):
    df = joblib.load(f"Penilaian_Model_LSTM_{i}.pkl")
    dfs.append(df)

df_lstm = pd.concat(dfs, axis=0, ignore_index=True)

df_lstm["QUOTE_DATE"] = pd.to_datetime(df_lstm["QUOTE_DATE"])
df_lstm = df_lstm.sort_values(["ID_OPTION", "QUOTE_DATE"]).reset_index(drop=True)

In [4]:
# MENGGABUNGKAN HASIL PENILAIAN HARGA OPSI PUT KEDUA MODEL
df_bs["QUOTE_DATE"] = pd.to_datetime(df_bs["QUOTE_DATE"])

df_merge = pd.merge(
    df_lstm,
    df_bs,
    on=["ID_OPTION", "QUOTE_DATE"]
)

In [5]:
# MENGGABUNGKAN HASIL PENILAIAN HARGA OPSI PUT DENGAN DATA CLEANING
df = pd.read_csv("Data_Kuotasi_Opsi_Put_NVDA_Cleaning.csv")
df["QUOTE_DATE"] = df["QUOTE_DATE"].astype(str).str.strip()
df["QUOTE_DATE"] = pd.to_datetime(df["QUOTE_DATE"], errors='coerce')

df_final = pd.merge(
    df_merge,       
    df,      
    on=["ID_OPTION", "QUOTE_DATE"]
)

columns_to_keep = [
    'QUOTE_DATE',
    'ID_OPTION',
    'UNDERLYING_LAST',
    'EXPIRE_DATE',
    'DTE',
    'STRIKE',
    'MID_PRICE',
    'BS_PRICE',
    'LSTM_PRICE'
]
df_final = df_final[columns_to_keep]

In [6]:
# MENYIMPAN DATA HASIL PENILAIAN HARGA OPSI PUT
df_final.to_csv("Data_Penilaian_Opsi_Put_NVDA.csv", index=False)

In [7]:
# MENGHITUNG NILAI KESALAHAN 
df = pd.read_csv("Data_Penilaian_Opsi_Put_NVDA.csv")

rmse1 = np.sqrt(mean_squared_error(df["MID_PRICE"], df["BS_PRICE"]))
rmse2 = np.sqrt(mean_squared_error(df["MID_PRICE"], df["LSTM_PRICE"]))

mae1 = mean_absolute_error(df["MID_PRICE"], df["BS_PRICE"])
mae2 = mean_absolute_error(df["MID_PRICE"], df["LSTM_PRICE"])

print("Analisis Error Model Black-Scholes")
print("RMSE:", rmse1)
print("MAE :", mae1)

print("\nAnalisis Error Model LSTM")
print("RMSE:", rmse2)
print("MAE :", mae2)

Analisis Error Model Black-Scholes
RMSE: 7.3032922752470615
MAE : 3.6559976048175518

Analisis Error Model LSTM
RMSE: 5.100368241230198
MAE : 3.0969520297712494


In [8]:
# MEMBUAT KELOMPOK BERDASARKAN WAKTU HINGGA JATUH TEMPO
bins = [0, 30, 90, 180, 270, 360, 540, 730]
labels = ["1 bulan", "3 bulan", "6 bulan", "9 bulan", "1 tahun", "1.5 tahun", "2 tahun"]

df["DTE_GROUP"] = pd.cut(
    df["DTE"],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

In [9]:
# MENGHITUNG NILAI KESALAHAN BERDASARKAN KELOMPOK WAKTU HINGGA JATUH TEMPO
def hitung_error(group):
    y_true = group["MID_PRICE"]

    y_lstm = group["LSTM_PRICE"]
    
    # Prediksi Black-Scholes
    y_bs = group["BS_PRICE"]
    
    return pd.Series({
        # LSTM
        "RMSE_LSTM": np.sqrt(mean_squared_error(y_true, y_lstm)),
        "MAE_LSTM": mean_absolute_error(y_true, y_lstm),

        # Black-Scholes
        "RMSE_BS": np.sqrt(mean_squared_error(y_true, y_bs)),
        "MAE_BS": mean_absolute_error(y_true, y_bs),

        # Jumlah data
        "N_DATA": len(group)
    })

# Hitung per grup
hasil_per_dte = (
    df
    .groupby("DTE_GROUP", observed=True)
    .apply(hitung_error, include_groups=False)
    .reset_index()
)

hasil_per_dte.head(8)

,DTE_GROUP,RMSE_LSTM,MAE_LSTM,RMSE_BS,MAE_BS,N_DATA
0,1 bulan,5.165259,2.891343,1.753408,0.890703,115246.0
1,3 bulan,5.117828,3.014961,3.457987,2.284642,79873.0
2,6 bulan,5.302736,3.240384,6.030438,3.826534,82495.0
3,9 bulan,5.001487,3.186194,7.860109,4.810204,52209.0
4,1 tahun,4.545205,2.947221,10.294055,6.030967,27204.0
5,1.5 tahun,4.742100,3.071701,13.105610,7.552869,42521.0
6,2 tahun,5.328665,3.638650,12.499452,7.198203,34890.0
